In [ ]:
import json
from typing import Any, Dict, List, TypedDict

import arxiv
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import END, StateGraph

from AgentPrompts import CRITIC_PROMPT, ORCHESTRATOR_PROMPT, RESEARCHER_PROMPT, problem_statement


In [ ]:
class State(TypedDict, total=False):
    problem_statement: str
    domain: str
    task_type: str
    core_problem: str
    problem_summary: str
    keywords: List[str]
    possible_methods: List[str]
    tavily_queries: List[str]
    github_queries: List[str]
    paper_queries: List[str]
    fetched_papers: List[Dict[str, Any]]
    papers: List[Dict[str, Any]]
    evaluation_report: Dict[str, Any]
    iterations: int


In [ ]:
llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq", temperature=0.2)
json_parser = JsonOutputParser()


### LangGraph research workflow




In [ ]:
def orchestrator_node(state: State) -> Dict[str, Any]:
    """Analyze the problem statement and generate focused research queries."""
    print("Orchestrator running")

    previous_queries = state.get("paper_queries", [])
    critic_feedback = state.get("evaluation_report", {}).get("feedback", "")

    prompt = ChatPromptTemplate.from_template(ORCHESTRATOR_PROMPT)
    chain = prompt | llm | json_parser
    output = chain.invoke({
        "problem_statement": state["problem_statement"],
        "previous_queries": previous_queries,
        "critic_feedback": critic_feedback,
    })

    return {
        "domain": output.get("domain", ""),
        "task_type": output.get("task_type", ""),
        "core_problem": output.get("core_problem", ""),
        "problem_summary": output.get("problem_summary", ""),
        "keywords": output.get("keywords", []),
        "possible_methods": output.get("possible_methods", []),
        "tavily_queries": output.get("tavily_queries", []),
        "github_queries": output.get("github_queries", []),
        "paper_queries": output.get("paper_queries", []),
        "iterations": state.get("iterations", 0) + 1,
    }


In [ ]:
def research_node(state: State) -> Dict[str, Any]:
    """Fetch arXiv papers and normalize them with the LLM."""
    queries = state.get("paper_queries", [])
    print(f"Researcher fetching papers for {len(queries)} queries")

    retrieved_papers = []
    seen_titles = {paper.get("title", "").lower() for paper in state.get("fetched_papers", [])}
    client = arxiv.Client()

    for query in queries:
        try:
            search = arxiv.Search(
                query=query,
                max_results=3,
                sort_by=arxiv.SortCriterion.Relevance,
            )

            for result in client.results(search):
                title_key = result.title.lower()
                if title_key in seen_titles:
                    continue

                seen_titles.add(title_key)
                retrieved_papers.append({
                    "title": result.title,
                    "authors": [author.name for author in result.authors],
                    "abstract": result.summary.replace("\n", " "),
                    "categories": result.categories,
                    "arxiv_url": result.entry_id,
                    "source_query": query,
                })
        except Exception as exc:
            print(f"Error querying arXiv for '{query}': {exc}")

    all_fetched = state.get("fetched_papers", []) + retrieved_papers

    if not all_fetched:
        return {"fetched_papers": [], "papers": []}

    prompt = ChatPromptTemplate.from_template(RESEARCHER_PROMPT)
    chain = prompt | llm | json_parser
    output = chain.invoke({
        "problem_statement": state["problem_statement"],
        "problem_summary": state.get("problem_summary", ""),
        "domain": state.get("domain", ""),
        "task_type": state.get("task_type", ""),
        "keywords": state.get("keywords", []),
        "possible_methods": state.get("possible_methods", []),
        "raw_papers": json.dumps(all_fetched, indent=2),
    })

    return {
        "fetched_papers": all_fetched,
        "papers": output.get("papers", []),
    }


In [ ]:
def critic_node(state: State) -> Dict[str, Any]:
    """Score the retained papers and produce feedback for retry routing."""
    print("Critic running")

    papers = state.get("papers", [])
    if not papers:
        return {
            "evaluation_report": {
                "relevance_score": 0,
                "feedback": "No relevant papers were retained. Use broader financial time-series and volatility forecasting terms.",
            }
        }

    prompt = ChatPromptTemplate.from_template(CRITIC_PROMPT)
    chain = prompt | llm | json_parser
    evaluation = chain.invoke({
        "problem_statement": state["problem_statement"],
        "problem_summary": state.get("problem_summary", ""),
        "papers": json.dumps(papers[:8], indent=2),
    })

    print(f"Score awarded: {evaluation.get('relevance_score', 0)}/100")
    return {"evaluation_report": evaluation}


In [ ]:
def should_continue(state: State) -> str:
    """Retry when the paper set is weak, with a hard cap to avoid infinite loops."""
    report = state.get("evaluation_report", {})
    score = report.get("relevance_score", 0)
    iterations = state.get("iterations", 0)

    if score >= 80 or iterations >= 5:
        if iterations >= 5 and score < 80:
            print("Stopping after 5 iterations without clearing the threshold.")
        else:
            print("Done")
        return "end"

    print(f"[System]: Relevance score ({score}) too low. Feedback: {report.get('feedback', '')}")
    print("Routing back to orchestrator")
    return "retry"


In [ ]:
workflow = StateGraph(State)

workflow.add_node("orchestrator", orchestrator_node)
workflow.add_node("researcher", research_node)
workflow.add_node("critic", critic_node)

workflow.set_entry_point("orchestrator")
workflow.add_edge("orchestrator", "researcher")
workflow.add_edge("researcher", "critic")
workflow.add_conditional_edges(
    "critic",
    should_continue,
    {"retry": "orchestrator", "end": END},
)

app = workflow.compile()


In [ ]:
initial_state: State = {
    "problem_statement": problem_statement,
    "fetched_papers": [],
    "papers": [],
    "evaluation_report": {},
    "iterations": 0,
}

final_output = app.invoke(initial_state)
final_output
